# Hot-pixel filtering and de-noising using IMC-Denoise

In [1]:
from IMC_Denoise.IMC_Denoise_main import DIMR, DeepSNiF
import tifffile
import os

In [5]:
img_folder = "../data/cell_mode/img_raw"
output_folder = "../data/cell_mode/img_dimr"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

For each channel we perform hot-pixel filtering using dimr. Since in channel 13 containing Ido1 we have low s/n ratio, we additionally correct with deepsnif.

In [6]:
model_weights_path = "../data/IMC_denoise_model_weights"
channel_to_denoise = 13


In [9]:
deepsnif = DeepSNiF.DeepSNiF(weights_name = f"IMC_Model_Channel_{channel_to_denoise}.keras",
                  weights_dir = model_weights_path, 
                  is_load_weights = True)

In [10]:
dimr = DIMR.DIMR(window_size=5)

img_files = [f for f in os.listdir(img_folder) if f.endswith('.tiff')]
for n, img_file in enumerate(img_files):
    img_path = os.path.join(img_folder, img_file)
    img = tifffile.imread(img_path)
    for ch_n, ch in enumerate(img):
        if ch_n == channel_to_denoise:
            img_channel = img[ch_n]  # Work directly with the loaded numpy array
            img_dimr_deepsnif = deepsnif.perform_IMC_Denoise(img_channel, n_neighbours=4, n_iter=3, window_size=3)
            img[ch_n] = img_dimr_deepsnif  # Update the channel in the numpy array
        else:
            img_channel = img[ch_n]
            img_dimr = dimr.perform_DIMR(img_channel)
            img[ch_n] = img_dimr
    output_path = os.path.join(output_folder, img_file)
    tifffile.imwrite(output_path, img)
    print(f"processed image nb. {n}, {img_file}")

processed image nb. 0, Patient1_002.tiff
processed image nb. 1, Patient1_003.tiff
processed image nb. 2, Patient2_001.tiff
processed image nb. 3, Patient3_001.tiff
processed image nb. 4, Patient4_006.tiff
processed image nb. 5, Patient4_007.tiff
processed image nb. 6, Patient4_008.tiff
processed image nb. 7, Patient3_003.tiff
processed image nb. 8, Patient2_003.tiff
processed image nb. 9, Patient3_002.tiff
processed image nb. 10, Patient2_002.tiff
processed image nb. 11, Patient4_005.tiff
processed image nb. 12, Patient1_001.tiff
processed image nb. 13, Patient2_004.tiff
